# Lumis 14: One-time test of model v2 (and v1 on the same phone photos)

- **SCIN test split** (phone photos, one close-up per case): never used before. v1 and v2 side by side, with a paired
  bootstrap for the difference. This is the main result.
- **DermNet test split:** already used once for v1 (notebook 08). Running v2 on it is a **second use**, reported as
  such; v1's 65.5% stays the clean DermNet figure. Nothing is chosen from these numbers.
- Each model uses its own calibration (v1: notebook 04; v2: notebook 13). Scabies cap on for both.

Runs once: refuses to run again if results exist. CPU runtime is fine, about 10 minutes.

In [ ]:
RUN_V2 = None      # None = latest v2 run
B = 2000
SEED = 42
CAP_SCABIES = True

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/Lumis'

In [ ]:
import os, json, zipfile, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

OUT = f'{BASE}/exports/final_test_v2'
if os.path.exists(f'{OUT}/metrics.json'):
    raise SystemExit(f'The v2 test was already run: see {OUT}. It must only be used once.')
os.makedirs(OUT, exist_ok=True)
for z, dest in [(f'{BASE}/data/dermnet.zip', '/content/dermnet'), (f'{BASE}/data/scin_images.zip', '/content')]:
    marker = '/content/dermnet' if 'dermnet' in z else '/content/scin'
    if not os.path.exists(marker):
        with zipfile.ZipFile(z) as f:
            f.extractall(dest)

runs = pd.read_csv(f'{BASE}/exports/runs_log.csv')
if RUN_V2 is None:
    RUN_V2 = runs[runs.split == 'v2+scin'].iloc[-1]['run']
v2 = runs[runs.run == RUN_V2].iloc[-1]
RUN_V1 = v2.base_run
v1 = runs[runs.run == RUN_V1].iloc[-1]
cal = {'v1': json.load(open(f'{BASE}/exports/runs/{RUN_V1}/calibration.json')),
       'v2': json.load(open(f'{BASE}/exports/runs/{RUN_V2}/calibration.json'))}
assert cal['v2'].get('accepted'), 'v2 was not accepted in notebook 13'
CLASSES = cal['v1']['classes']
K = len(CLASSES)

scin = pd.read_csv(f'{BASE}/data/scin_split_v1.csv', dtype={'case_id': str})
s_test = scin[(scin.split == 'test') & scin.eval_photo].reset_index(drop=True)
derm = pd.read_csv(f'{BASE}/data/lumis_split_v2.csv')
d_test = derm[derm.split == 'test'].reset_index(drop=True)
d_test = d_test.merge(pd.read_csv(f'{BASE}/data/skin_tone_labels_round1.csv')[['path', 'tone']], on='path', how='left')
print(f'v1 {RUN_V1} | v2 {RUN_V2} | SCIN test {len(s_test)} cases | DermNet test {len(d_test)} photos')
print('SCIN test by condition:', s_test.lumis_class.value_counts().to_dict())

## 1. Predictions

In [ ]:
import tensorflow as tf
import keras

def load(path):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.cast(tf.cast(tf.image.resize(img, (224, 224)), tf.uint8), tf.float32)
    return keras.applications.efficientnet.preprocess_input(img)

def predict(model, paths):
    ds = tf.data.Dataset.from_tensor_slices(paths).map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(32)
    return model.predict(ds, verbose=0)

s_paths = [os.path.join('/content', p) for p in s_test.path]
d_paths = [os.path.join('/content/dermnet', p) for p in d_test.path]
m1, m2 = keras.models.load_model(v1.model_file), keras.models.load_model(v2.model_file)
t0 = time.time()
logits = {('v1', 'SCIN'): predict(m1, s_paths), ('v2', 'SCIN'): predict(m2, s_paths),
          ('v1', 'DermNet'): np.load(f'{BASE}/exports/final_test/test_logits.npy'), ('v2', 'DermNet'): predict(m2, d_paths)}
assert len(logits[('v1', 'DermNet')]) == len(d_test)
print(f'Predicted in {time.time() - t0:.0f} s')
for (m, d), l in logits.items():
    np.save(f'{OUT}/logits_{m}_{d}.npy', l)

## 2. Results

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)     # empty bands give NaN, reported as nan
from sklearn.metrics import f1_score, confusion_matrix

def softmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)

def apply(model, lg):
    c = cal[model]
    probs = softmax((lg + np.array(c['logit_adjust'])) / c['temperature'])
    pred, conf = probs.argmax(1), probs.max(1)
    b = c['bands']
    band = np.where(conf >= b['high_from'], 'high', np.where(conf >= b['refer_below'], 'moderate', 'refer'))
    if CAP_SCABIES:
        band = np.where((pred == CLASSES.index('scabies')) & (band == 'high'), 'moderate', band)
    return pred, conf, band

y = {'SCIN': np.array([CLASSES.index(c) for c in s_test.lumis_class]),
     'DermNet': np.array([CLASSES.index(c) for c in d_test.lumis_class])}
res = {k: apply(k[0], l) for k, l in logits.items()}

def summary(yt, yp, band):
    cm = confusion_matrix(yt, yp, labels=range(K))
    present = cm.sum(1) > 0
    sens = (cm.diagonal() / np.maximum(cm.sum(1), 1))[present]
    answered = band != 'refer'
    return {'accuracy': (yt == yp).mean(), 'macro_sensitivity': sens.mean(),
            'macro_f1': f1_score(yt, yp, labels=np.where(present)[0], average='macro', zero_division=0),
            'referred': 1 - answered.mean(),
            'accuracy_when_answering': (yt[answered] == yp[answered]).mean() if answered.any() else np.nan,
            'high_share': (band == 'high').mean(),
            'high_accuracy': (yt[band == 'high'] == yp[band == 'high']).mean() if (band == 'high').any() else np.nan}

rng = np.random.default_rng(SEED)
boot_idx = {d: [rng.integers(0, len(y[d]), len(y[d])) for _ in range(B)] for d in y}

def with_ci(model, d, idx=None):
    yt, (yp, _, band) = y[d], res[(model, d)]
    sel = np.arange(len(yt)) if idx is None else idx
    point = summary(yt[sel], yp[sel], band[sel])
    boots = [summary(yt[sel][i], yp[sel][i], band[sel][i]) for i in [rng.integers(0, len(sel), len(sel)) for _ in range(B)]]
    return {k: (round(float(v), 4), [round(float(np.nanpercentile([b[k] for b in boots], q)), 4) for q in (2.5, 97.5)])
            for k, v in point.items()}

results = {f'{m} {d}': with_ci(m, d) for m in ['v1', 'v2'] for d in ['SCIN', 'DermNet']}
table = pd.DataFrame({name: {k: f'{v:.3f} ({ci[0]:.3f}-{ci[1]:.3f})' for k, (v, ci) in r.items()} for name, r in results.items()})
display(table[['v1 SCIN', 'v2 SCIN', 'v1 DermNet', 'v2 DermNet']])
print('DermNet test: second use for v2 (v1 figure is the clean one).')

In [ ]:
# Paired difference v2 - v1 on the SCIN test cases (same resampled cases for both models)
diffs = {k: [] for k in ['accuracy', 'macro_sensitivity', 'accuracy_when_answering']}
for i in boot_idx['SCIN']:
    a = summary(y['SCIN'][i], res[('v1', 'SCIN')][0][i], res[('v1', 'SCIN')][2][i])
    b = summary(y['SCIN'][i], res[('v2', 'SCIN')][0][i], res[('v2', 'SCIN')][2][i])
    for k in diffs:
        diffs[k].append(b[k] - a[k])
p1 = summary(y['SCIN'], res[('v1', 'SCIN')][0], res[('v1', 'SCIN')][2])
p2 = summary(y['SCIN'], res[('v2', 'SCIN')][0], res[('v2', 'SCIN')][2])
paired = {k: {'v2_minus_v1': round(float(p2[k] - p1[k]), 4),
              'ci95': [round(float(np.nanpercentile(v, q)), 4) for q in (2.5, 97.5)]} for k, v in diffs.items()}
for k, v in paired.items():
    verdict = 'interval excludes 0' if v['ci95'][0] > 0 or v['ci95'][1] < 0 else 'interval includes 0'
    print(f"{k:24s} {v['v2_minus_v1']:+.3f}  (95% CI {v['ci95'][0]:+.3f} to {v['ci95'][1]:+.3f})  {verdict}")

In [ ]:
def per_class(model, d):
    yt, (yp, _, _) = y[d], res[(model, d)]
    cm = confusion_matrix(yt, yp, labels=range(K))
    rows = {}
    for i, c in enumerate(CLASSES):
        tp, fn, fp = cm[i, i], cm[i].sum() - cm[i, i], cm[:, i].sum() - cm[i, i]
        rows[c] = {'n': int(cm[i].sum()), 'sensitivity': round(tp / (tp + fn), 3) if tp + fn else None,
                   'precision': round(tp / (tp + fp), 3) if tp + fp else None}
    return pd.DataFrame(rows).T

pc = pd.concat({f'{m} {d}': per_class(m, d) for m in ['v1', 'v2'] for d in ['SCIN', 'DermNet']}, axis=1)
display(pc)

def bands_table(model, d):
    yt, (yp, _, band) = y[d], res[(model, d)]
    return pd.DataFrame({b: {'share': round((band == b).mean(), 3),
                             'accuracy': round((yt[band == b] == yp[band == b]).mean(), 3) if (band == b).any() else None}
                         for b in ['high', 'moderate', 'refer']}).T
bt = pd.concat({f'{m} {d}': bands_table(m, d) for m in ['v1', 'v2'] for d in ['SCIN', 'DermNet']}, axis=1)
display(bt)

def tone_groups(d):
    t = (s_test if d == 'SCIN' else d_test).tone
    return {'I-II': np.where(t == 'I-II')[0], 'III-VI': np.where(t.isin(['III-IV', 'V-VI']))[0]}
tone_rows = []
for m in ['v1', 'v2']:
    for d in ['SCIN', 'DermNet']:
        for g, idx in tone_groups(d).items():
            if len(idx):
                r = with_ci(m, d, idx)
                tone_rows.append({'model': m, 'data': d, 'skin type': g, 'n': len(idx),
                                  'accuracy': r['accuracy'][0], 'accuracy_ci': r['accuracy'][1],
                                  'referred': r['referred'][0]})
tones = pd.DataFrame(tone_rows)
display(tones)

## 3. Save

In [ ]:
for (m, d), (yp, conf, band) in res.items():
    src = s_test if d == 'SCIN' else d_test
    pd.DataFrame({'path': src.path, 'true': src.lumis_class, 'pred': [CLASSES[i] for i in yp],
                  'confidence': conf.round(4), 'band': band, 'correct': yp == y[d], 'tone': src.tone}
                 ).to_csv(f'{OUT}/predictions_{m}_{d}.csv', index=False)
pc.to_csv(f'{OUT}/per_class.csv'); bt.to_csv(f'{OUT}/bands.csv'); tones.to_csv(f'{OUT}/by_skin_type.csv', index=False)
metrics = {'evaluated_at': time.strftime('%Y-%m-%d %H:%M'), 'run_v1': RUN_V1, 'run_v2': RUN_V2,
           'n_scin_test_cases': int(len(s_test)), 'n_dermnet_test': int(len(d_test)),
           'scin_test_per_class': s_test.lumis_class.value_counts().to_dict(),
           'dermnet_test_note': 'second use of the DermNet test split (v2); v1 DermNet figures are from notebook 08',
           'results': results, 'paired_scin_v2_minus_v1': paired, 'scabies_cap': CAP_SCABIES,
           'calibration_v2': {k: cal['v2'][k] for k in ['temperature', 'prior_tau', 'bands']}}
json.dump(metrics, open(f'{OUT}/metrics.json', 'w'), indent=1, default=float)
print('Saved to', OUT)